# P8-HAZARD-SCHEMA-DESIGN — External Hazard Data Integration Schema

**Author:** @data-planner | **Date:** 2026-10-21 | **Schema:** `cdm_tmforum.provable_resilience_intelligence`

## Purpose
Define SID-aligned staging tables for 5 external hazard data feeds that enrich the existing `geographic_hazard_event` table (50,080 rows). The staging layer captures source-native fields, the enrichment crosswalk provides spatial joins to `physical_resource_span`, and the refresh pipeline keeps resilience scores current.

## Architecture
```
External APIs                    Staging (Bronze)              Enrichment (Silver)         Gold
─────────────                    ────────────────              ───────────────────         ────
api.weather.gov/alerts      →  stg_noaa_weather_alert    ┐
earthquake.usgs.gov         →  stg_usgs_earthquake       │
waterservices.usgs.gov      →  stg_usgs_flood_warning    ├→ hazard_span_intersection → geographic_hazard_event
NIFC Open Data (WFIGS)      →  stg_nifc_wildfire         │                              (existing, enriched)
State 811 APIs              →  stg_811_dig_ticket        ┘
```

## Branding: All assets use **Lakelink Fiber** (never Lumen).
## SID Domain: TMF SID Common — Geographic Place / Hazard Event extension

In [0]:
%sql
-- ============================================================================
-- TABLE 1: stg_noaa_weather_alert
-- Source: NWS Weather Alerts API (api.weather.gov/alerts)
-- Refresh: Every 15 minutes (streaming-capable, alerts change rapidly)
-- SID Concept: GeographicPlace → HazardEvent (severe weather extension)
-- Volume: ~200-2,000 active alerts nationwide at any time; ~50K/year archived
-- ============================================================================

CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.stg_noaa_weather_alert (
  alert_id              STRING        NOT NULL  COMMENT 'NWS alert URI (e.g. urn:oid:2.49.0.1.840.0...). Globally unique.',
  event_type            STRING        NOT NULL  COMMENT 'NWS event name: Tornado Warning, Severe Thunderstorm Warning, Winter Storm Warning, Flood Warning, etc.',
  severity              STRING        NOT NULL  COMMENT 'CAP severity: Extreme | Severe | Moderate | Minor | Unknown',
  certainty             STRING        NOT NULL  COMMENT 'CAP certainty: Observed | Likely | Possible | Unlikely | Unknown',
  urgency               STRING        NOT NULL  COMMENT 'CAP urgency: Immediate | Expected | Future | Past | Unknown',
  response_type         STRING                  COMMENT 'CAP response: Shelter | Evacuate | Prepare | Execute | Avoid | Monitor | None',
  onset                 TIMESTAMP     NOT NULL  COMMENT 'When the alert hazard begins (UTC)',
  expires               TIMESTAMP               COMMENT 'When the alert expires (UTC). NULL if open-ended.',
  effective             TIMESTAMP               COMMENT 'When the alert information becomes effective (UTC)',
  sent                  TIMESTAMP     NOT NULL  COMMENT 'When the alert was issued by NWS (UTC)',
  geom_wkt              STRING        NOT NULL  COMMENT 'WKT POLYGON or MULTIPOLYGON from GeoJSON geometry. WGS84.',
  centroid_lat          DOUBLE                  COMMENT 'Centroid latitude computed from geom_wkt for proximity queries',
  centroid_lon          DOUBLE                  COMMENT 'Centroid longitude computed from geom_wkt for proximity queries',
  affected_zones        STRING                  COMMENT 'Comma-separated NWS zone codes (e.g. TXZ104,TXZ105)',
  area_desc             STRING                  COMMENT 'Human-readable area description from NWS',
  headline              STRING                  COMMENT 'NWS headline text for the alert',
  description           STRING                  COMMENT 'Full NWS alert description text',
  instruction           STRING                  COMMENT 'NWS recommended protective actions',
  sender_name           STRING                  COMMENT 'Issuing NWS Weather Forecast Office name',
  source_url            STRING        NOT NULL  COMMENT 'Full API URL that returned this alert',
  ingested_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when row was ingested into staging',
  raw_json              STRING                  COMMENT 'Original GeoJSON feature (for audit/replay)',
  is_active             BOOLEAN       NOT NULL  COMMENT 'TRUE if alert is currently active (expires > NOW())'
)
COMMENT 'Staging: NWS severe weather alerts from api.weather.gov. One row per alert. Refreshed every 15 min. Source for geographic_hazard_event severe_weather enrichment. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'bronze',
  'refresh.cadence' = 'PT15M',
  'source.api' = 'api.weather.gov/alerts',
  'sid.concept' = 'Common.GeographicPlace.HazardEvent'
);

In [0]:
%sql
-- ============================================================================
-- TABLE 2: stg_usgs_earthquake
-- Source: USGS Earthquake Hazards API (earthquake.usgs.gov/fdsnws/event/1)
-- Refresh: Every 30 minutes
-- SID Concept: GeographicPlace → HazardEvent (seismic extension)
-- Volume: ~20K events/year globally; ~2K M2.5+ in CONUS
-- ============================================================================

CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.stg_usgs_earthquake (
  event_id              STRING        NOT NULL  COMMENT 'USGS event identifier (e.g. us7000abcd). Globally unique.',
  magnitude             DOUBLE        NOT NULL  COMMENT 'Earthquake magnitude on the scale indicated by mag_type',
  mag_type              STRING        NOT NULL  COMMENT 'Magnitude type: ml (local), mb (body), mw (moment), ms (surface)',
  event_type            STRING        NOT NULL  COMMENT 'Event type: earthquake | quarry blast | explosion | ice quake',
  alert_level           STRING                  COMMENT 'PAGER alert level: green | yellow | orange | red. NULL if not computed.',
  significance          INT                     COMMENT 'USGS significance score (0-1000). Higher = more significant.',
  latitude              DOUBLE        NOT NULL  COMMENT 'WGS84 latitude of earthquake epicenter',
  longitude             DOUBLE        NOT NULL  COMMENT 'WGS84 longitude of earthquake epicenter',
  depth_km              DOUBLE        NOT NULL  COMMENT 'Depth of earthquake hypocenter in kilometers',
  place_description     STRING                  COMMENT 'Human-readable location (e.g. 15 km NNE of Oklahoma City)',
  geom_wkt              STRING        NOT NULL  COMMENT 'WKT POINT geometry from GeoJSON coordinates. WGS84.',
  tsunami_flag          INT                     COMMENT 'Tsunami advisory: 0 = no, 1 = yes',
  felt_reports          INT                     COMMENT 'Number of DYFI (Did You Feel It?) reports received',
  cdi                   DOUBLE                  COMMENT 'Community Determined Intensity (Mercalli scale, 1-10)',
  mmi                   DOUBLE                  COMMENT 'Maximum Modified Mercalli Intensity (ShakeMap)',
  event_time            TIMESTAMP     NOT NULL  COMMENT 'Origin time of the earthquake (UTC, from UNIX epoch ms)',
  updated_time          TIMESTAMP               COMMENT 'Last update time from USGS (UTC)',
  review_status         STRING                  COMMENT 'Review status: automatic | reviewed | deleted',
  estimated_radius_km   DOUBLE                  COMMENT 'Estimated impact radius based on magnitude: 10^(0.5*mag-1.0) km. For buffer zone spatial join.',
  source_url            STRING        NOT NULL  COMMENT 'USGS detail GeoJSON URL for this event',
  ingested_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when row was ingested into staging',
  raw_json              STRING                  COMMENT 'Original GeoJSON feature (for audit/replay)'
)
COMMENT 'Staging: USGS earthquake events from FDSNWS API. One row per seismic event. Refreshed every 30 min. Source for geographic_hazard_event earthquake enrichment. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'bronze',
  'refresh.cadence' = 'PT30M',
  'source.api' = 'earthquake.usgs.gov/fdsnws/event/1',
  'sid.concept' = 'Common.GeographicPlace.HazardEvent'
);

In [0]:
%sql
-- ============================================================================
-- TABLE 3: stg_usgs_flood_warning
-- Source: NWS Alerts API (flood types) + USGS Water Services (river gauges)
-- Refresh: Every 30 minutes
-- SID Concept: GeographicPlace → HazardEvent (hydrological extension)
-- Volume: ~500-5,000 active flood products; ~10K/year archived
-- ============================================================================

CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.stg_usgs_flood_warning (
  warning_id            STRING        NOT NULL  COMMENT 'NWS alert URI for flood warning. Globally unique.',
  flood_category        STRING        NOT NULL  COMMENT 'NWS flood category: major | moderate | minor | action | no_flooding',
  event_type            STRING        NOT NULL  COMMENT 'NWS event: Flood Warning | Flash Flood Warning | River Flood Warning | Coastal Flood Warning',
  severity              STRING        NOT NULL  COMMENT 'CAP severity: Extreme | Severe | Moderate | Minor',
  gauge_id              STRING                  COMMENT 'USGS gauge station ID (e.g. 07010000). NULL for areal flood warnings.',
  gauge_name            STRING                  COMMENT 'USGS gauge station name (e.g. Mississippi River at St. Louis)',
  observed_stage_ft     DOUBLE                  COMMENT 'Most recent observed river stage in feet',
  flood_stage_ft        DOUBLE                  COMMENT 'NWS-defined flood stage threshold in feet for this gauge',
  forecast_crest_ft     DOUBLE                  COMMENT 'Forecast crest stage in feet. NULL if not forecast.',
  forecast_crest_time   TIMESTAMP               COMMENT 'Forecast time of crest (UTC)',
  latitude              DOUBLE        NOT NULL  COMMENT 'WGS84 latitude of gauge or warning area centroid',
  longitude             DOUBLE        NOT NULL  COMMENT 'WGS84 longitude of gauge or warning area centroid',
  geom_wkt              STRING                  COMMENT 'WKT geometry: POINT for gauge-based, POLYGON for areal flood warnings. WGS84.',
  area_desc             STRING                  COMMENT 'Human-readable flood area description',
  onset                 TIMESTAMP     NOT NULL  COMMENT 'When the flood warning begins (UTC)',
  expires               TIMESTAMP               COMMENT 'When the flood warning expires (UTC)',
  observed_time         TIMESTAMP               COMMENT 'Timestamp of most recent stage observation (UTC)',
  headline              STRING                  COMMENT 'NWS flood warning headline',
  description           STRING                  COMMENT 'Full warning text including stage forecasts',
  estimated_radius_km   DOUBLE                  COMMENT 'Estimated flood impact radius: 5km minor, 10km moderate, 20km major. For spatial join to spans.',
  source_url            STRING        NOT NULL  COMMENT 'API URL that returned this warning',
  ingested_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when row was ingested into staging',
  raw_json              STRING                  COMMENT 'Original JSON payload (for audit/replay)',
  is_active             BOOLEAN       NOT NULL  COMMENT 'TRUE if warning is currently active'
)
COMMENT 'Staging: NWS flood warnings + USGS river gauge data. One row per flood warning. Refreshed every 30 min. Source for geographic_hazard_event flood enrichment. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'bronze',
  'refresh.cadence' = 'PT30M',
  'source.api' = 'api.weather.gov/alerts + waterservices.usgs.gov',
  'sid.concept' = 'Common.GeographicPlace.HazardEvent'
);

In [0]:
%sql
-- ============================================================================
-- TABLE 4: stg_nifc_wildfire
-- Source: NIFC Open Data WFIGS Interagency Fire Perimeters
-- Refresh: Every 2 hours (perimeters update as fires evolve)
-- SID Concept: GeographicPlace → HazardEvent (wildfire extension)
-- Volume: ~200-500 active incidents in peak season; ~60K perimeters/year
-- ============================================================================

CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.stg_nifc_wildfire (
  irwin_id              STRING        NOT NULL  COMMENT 'IRWIN (Integrated Reporting of Wildland-Fire Information) unique fire ID',
  perimeter_id          STRING        NOT NULL  COMMENT 'Unique perimeter feature ID from WFIGS. One fire may have multiple perimeters over time.',
  fire_name             STRING        NOT NULL  COMMENT 'Official incident name (e.g. Park Fire, Bridge Fire)',
  fire_type             STRING                  COMMENT 'Incident type: Wildfire | Prescribed Fire | Complex',
  fire_cause            STRING                  COMMENT 'Cause category: Human | Lightning | Unknown | Under Investigation',
  incident_status       STRING                  COMMENT 'Incident status: Active | Contained | Controlled | Out',
  containment_pct       DOUBLE                  COMMENT 'Percent contained (0-100). NULL if not reported.',
  acres_burned          DOUBLE                  COMMENT 'GIS-calculated acres within the fire perimeter',
  discovery_acres       DOUBLE                  COMMENT 'Acres at time of discovery/first report',
  perimeter_wkt         STRING        NOT NULL  COMMENT 'WKT POLYGON or MULTIPOLYGON of fire perimeter boundary. WGS84.',
  centroid_lat          DOUBLE        NOT NULL  COMMENT 'Centroid latitude of fire perimeter for proximity queries',
  centroid_lon          DOUBLE        NOT NULL  COMMENT 'Centroid longitude of fire perimeter for proximity queries',
  estimated_radius_km   DOUBLE                  COMMENT 'Derived: sqrt(acres_burned * 0.00404686 / PI) km. Circular-equivalent radius for buffer zone.',
  discovery_date        TIMESTAMP     NOT NULL  COMMENT 'Fire discovery or first-report datetime (UTC)',
  containment_date      TIMESTAMP               COMMENT 'Date fire reached full containment (UTC). NULL if ongoing.',
  perimeter_date        TIMESTAMP     NOT NULL  COMMENT 'Date/time this perimeter was captured (UTC)',
  personnel_count       INT                     COMMENT 'Number of personnel assigned to the incident',
  structures_threatened INT                     COMMENT 'Number of structures threatened',
  structures_destroyed  INT                     COMMENT 'Number of structures destroyed',
  source_url            STRING        NOT NULL  COMMENT 'NIFC WFIGS feature service URL',
  ingested_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when row was ingested into staging',
  raw_json              STRING                  COMMENT 'Original GeoJSON feature (for audit/replay)'
)
COMMENT 'Staging: NIFC active wildfire perimeters from WFIGS Open Data. One row per perimeter snapshot. Refreshed every 2 hours. Source for geographic_hazard_event wildfire enrichment. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'bronze',
  'refresh.cadence' = 'PT2H',
  'source.api' = 'data-nifc.opendata.arcgis.com/WFIGS',
  'sid.concept' = 'Common.GeographicPlace.HazardEvent'
);

In [0]:
%sql
-- ============================================================================
-- TABLE 5: stg_811_dig_ticket
-- Source: State 811 One-Call systems (DigAlert, MISS DIG, USIC, etc.)
-- Refresh: Every 6 hours (batch; 811 systems are not real-time)
-- SID Concept: Common.GeographicPlace → HazardEvent (excavation extension)
-- Volume: ~800K tickets/year nationwide; ~30K in Lakelink Fiber territory
-- Reference: copper_retirement.dig_safe_incident schema pattern
-- ============================================================================

CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.stg_811_dig_ticket (
  ticket_number         STRING        NOT NULL  COMMENT 'State 811 ticket number (format varies by state, e.g. 2026101500123)',
  revision_number       INT           NOT NULL  COMMENT 'Ticket revision number (re-marks, updates). 0 = original.',
  state                 STRING        NOT NULL  COMMENT 'US state abbreviation (e.g. TX, CA, CO)',
  county                STRING                  COMMENT 'County name where excavation is planned',
  city                  STRING                  COMMENT 'City or municipality name',
  street                STRING                  COMMENT 'Street address or intersection of planned excavation',
  cross_street          STRING                  COMMENT 'Nearest cross street for location reference',
  latitude              DOUBLE        NOT NULL  COMMENT 'WGS84 latitude of dig site (geocoded from address or GPS)',
  longitude             DOUBLE        NOT NULL  COMMENT 'WGS84 longitude of dig site',
  geom_wkt              STRING        NOT NULL  COMMENT 'WKT POINT geometry from lat/lon. WGS84.',
  work_type             STRING                  COMMENT 'Type of work: excavation | boring | trenching | grading | demolition | directional_drill',
  caller_type           STRING                  COMMENT 'Caller category: contractor | homeowner | utility | government | engineer',
  excavation_method     STRING                  COMMENT 'Method: mechanical | hand_dig | vacuum | hydro_excavation',
  excavation_depth_ft   DOUBLE                  COMMENT 'Planned excavation depth in feet. Critical for fiber depth-of-cover analysis.',
  utility_type          STRING                  COMMENT 'Utility types at risk: fiber | copper | gas | electric | water | sewer | telecom',
  work_date             DATE          NOT NULL  COMMENT 'Planned start date of excavation work',
  response_due          TIMESTAMP               COMMENT 'Deadline for utility locate response (typically 48-72 hours)',
  ticket_created        TIMESTAMP     NOT NULL  COMMENT 'When the 811 ticket was created',
  ticket_expires        TIMESTAMP               COMMENT 'When the ticket authorization expires',
  ticket_status         STRING        NOT NULL  COMMENT 'Ticket status: open | marked | clear | expired | cancelled',
  emergency_flag        BOOLEAN       NOT NULL  COMMENT 'TRUE if emergency excavation (bypasses normal wait period)',
  one_call_compliant    BOOLEAN                 COMMENT 'TRUE if proper 811 notification was filed before dig',
  estimated_radius_km   DOUBLE                  COMMENT 'Estimated impact radius: 0.05km (50m) for point excavation, 0.5km for linear projects',
  source_system         STRING        NOT NULL  COMMENT '811 system: digalert | miss_dig | usic | one_call_concepts | utility_notification_center',
  source_url            STRING                  COMMENT 'API endpoint or batch file source',
  ingested_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when row was ingested into staging',
  raw_json              STRING                  COMMENT 'Original ticket payload (for audit/replay)'
)
COMMENT 'Staging: State 811 One-Call dig tickets. One row per ticket revision. Refreshed every 6 hours (batch). Source for geographic_hazard_event dig_ticket enrichment. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'bronze',
  'refresh.cadence' = 'PT6H',
  'source.api' = 'State 811 systems (DigAlert, MISS DIG, etc.)',
  'sid.concept' = 'Common.GeographicPlace.HazardEvent'
);

In [0]:
%sql
-- ============================================================================
-- TABLE 6: hazard_span_intersection (Silver enrichment crosswalk)
-- Purpose: Spatial join between ANY hazard and physical_resource_span
-- Refresh: After each staging table refresh (event-driven)
-- Join Logic: Haversine from hazard centroid to span midpoint within radius_km
-- ============================================================================

CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection (
  intersection_id       STRING        NOT NULL  COMMENT 'Deterministic ID: SHA2(hazard_source || hazard_source_id || span_id, 256)',
  hazard_source         STRING        NOT NULL  COMMENT 'Staging source: noaa_weather | usgs_earthquake | usgs_flood | nifc_wildfire | 811_dig_ticket',
  hazard_source_id      STRING        NOT NULL  COMMENT 'PK from staging table (alert_id, event_id, warning_id, irwin_id, ticket_number)',
  hazard_type           STRING        NOT NULL  COMMENT 'Normalized type: severe_weather | earthquake | flood | wildfire | dig_ticket',
  hazard_severity       STRING        NOT NULL  COMMENT 'Normalized severity: critical | high | medium | low (mapped from source-native scales)',
  hazard_lat            DOUBLE        NOT NULL  COMMENT 'Hazard centroid/epicenter latitude',
  hazard_lon            DOUBLE        NOT NULL  COMMENT 'Hazard centroid/epicenter longitude',
  hazard_radius_km      DOUBLE        NOT NULL  COMMENT 'Impact radius used for this intersection',
  hazard_start          TIMESTAMP     NOT NULL  COMMENT 'Hazard event start time (UTC)',
  hazard_end            TIMESTAMP               COMMENT 'Hazard event end time (UTC). NULL if ongoing.',
  span_id               STRING        NOT NULL  COMMENT 'FK to physical_resource_span.span_id',
  span_midpoint_lat     DOUBLE        NOT NULL  COMMENT 'Span midpoint latitude (denormalized for query performance)',
  span_midpoint_lon     DOUBLE        NOT NULL  COMMENT 'Span midpoint longitude (denormalized for query performance)',
  distance_km           DOUBLE        NOT NULL  COMMENT 'Haversine distance from hazard centroid to span midpoint in km',
  is_within_radius      BOOLEAN       NOT NULL  COMMENT 'TRUE if distance_km <= hazard_radius_km',
  proximity_band        STRING        NOT NULL  COMMENT 'Distance band: direct_impact (<1km) | near_field (1-5km) | buffer_zone (5-radius_km)',
  affected_service_ids  STRING                  COMMENT 'Comma-separated service_ids routed through this span',
  affected_service_cnt  INT                     COMMENT 'Count of services routed through this span',
  computed_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when this intersection was computed',
  batch_id              STRING        NOT NULL  COMMENT 'Processing batch identifier for incremental refresh'
)
COMMENT 'Silver: Hazard-to-span spatial intersection crosswalk. One row per (hazard, span) pair within buffer radius. Bridges staging tables to physical_resource_span for resilience scoring. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'silver',
  'refresh.cadence' = 'event-driven',
  'sid.concept' = 'Common.GeographicPlace x Resource.PhysicalResourceSpan'
);

In [0]:
%sql
-- ============================================================================
-- SEVERITY MAPPING: Source-native → Normalized PRI scale
-- Used by hazard_span_intersection derivation pipeline
-- ============================================================================

-- NOAA Weather (CAP severity):  Extreme→critical, Severe→high, Moderate→medium, Minor/Unknown→low
-- USGS Earthquake (PAGER):      red→critical, orange→high, yellow→medium, green/NULL→low
--   fallback: mag>=6.0→high, mag>=4.5→medium, else low
-- NWS Flood (category):         major→critical, moderate→high, minor→medium, action→low
-- NIFC Wildfire (containment):  <10% AND >10Kac→critical, <30% AND >1Kac→high, <60%→medium, else low
-- 811 Dig (emergency+depth):    emergency→critical, depth>4ft→high, depth>2ft→medium, else low

SELECT 'Severity mapping rules documented for @data-engineer implementation' AS status;

In [0]:
%sql
-- ============================================================================
-- ENRICHMENT JOIN: Staging → geographic_hazard_event (MERGE upsert)
-- Template for @data-engineer to implement per staging source
-- ============================================================================

-- Example: NOAA Weather → geographic_hazard_event
/*
MERGE INTO cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event AS tgt
USING (
  SELECT 
    alert_id AS hazard_id,
    'severe_weather' AS hazard_type,
    CASE severity WHEN 'Extreme' THEN 'critical' WHEN 'Severe' THEN 'high'
                  WHEN 'Moderate' THEN 'medium' ELSE 'low' END AS severity,
    centroid_lat AS latitude, centroid_lon AS longitude,
    30.0 AS radius_km, geom_wkt,
    onset AS event_start, expires AS event_end,
    'nws' AS source,
    CONCAT(event_type, ': ', COALESCE(headline, '')) AS description,
    CURRENT_TIMESTAMP() AS created_at
  FROM cdm_tmforum.provable_resilience_intelligence.stg_noaa_weather_alert
  WHERE is_active = TRUE
) AS src ON tgt.hazard_id = src.hazard_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
*/

-- Spatial Join: Hazard → Span (populates hazard_span_intersection)
-- Uses bounding-box pre-filter + haversine for precision
/*
INSERT INTO cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection
SELECT SHA2(CONCAT(src, '|', hid, '|', sid), 256) AS intersection_id, ...
FROM geographic_hazard_event h
CROSS JOIN physical_resource_span s
WHERE s.midpoint_lat IS NOT NULL
  AND ABS(h.latitude - s.midpoint_lat) < (h.radius_km / 111.0)
  AND ABS(h.longitude - s.midpoint_lon) < (h.radius_km / (111.0 * COS(RADIANS(h.latitude))))
  AND <haversine> <= h.radius_km;
*/

SELECT 'Enrichment join patterns documented for @data-engineer' AS status;

## Refresh Cadence Summary

| Staging Table | Source API | Cadence | Priority |
|---|---|---|---|
| `stg_noaa_weather_alert` | api.weather.gov/alerts | 15 min | P0 — active severe weather |
| `stg_usgs_earthquake` | earthquake.usgs.gov/fdsnws | 30 min | P1 — seismic events |
| `stg_usgs_flood_warning` | api.weather.gov + waterservices.usgs.gov | 30 min | P1 — flood warnings |
| `stg_nifc_wildfire` | data-nifc.opendata.arcgis.com/WFIGS | 2 hours | P1 — perimeters evolve slowly |
| `stg_811_dig_ticket` | State 811 batch APIs | 6 hours | P2 — tickets are pre-planned |
| `hazard_span_intersection` | Silver derivation | After each staging refresh | P0 — feeds resilience score |

## Pipeline Dependency Chain
```
stg_noaa_weather_alert ─┐
stg_usgs_earthquake ────┤
stg_usgs_flood_warning ─┼→ geographic_hazard_event (MERGE) → hazard_span_intersection → resilience_score_snapshot
stg_nifc_wildfire ──────┤
stg_811_dig_ticket ─────┘
```

## Entity Relationship: Full Hazard Domain
```
stg_noaa_weather_alert ──┐
stg_usgs_earthquake ─────┤
stg_usgs_flood_warning ──┤── geographic_hazard_event (50K) ── hazard_span_intersection ── physical_resource_span (10K)
stg_nifc_wildfire ───────┤                                                                    │
stg_811_dig_ticket ──────┘                                                              service_path (54K)
                                                                                              │
                                                                                        service_inventory (8K)
                                                                                       /       |        \
                                                                              customer (1.2K)  score     proactive_action
```

## SID Domain Mapping (6 domains)
* **Common (GeographicPlace/HazardEvent):** All 5 staging + geographic_hazard_event + hazard_span_intersection
* **Resource (PhysicalResource):** physical_resource_span, shared_risk_group (spatial join target)
* **Service (ServiceInventory):** service_inventory, service_path (impact propagation)
* **Customer (Customer):** customer (SLA/revenue impact)
* **Product (ProductOffering):** service_inventory.service_type (indirect, P9 future enrichment)
* **Enterprise (BusinessInteraction):** proactive_action (response to hazard)